# Deep Learning Training Techniques

Best practices for training neural networks.

## Learning Objectives

- Understand regularization methods
- Learn about optimizers and learning rates
- Implement data augmentation
- Use learning rate schedules
- Apply early stopping and checkpointing

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Regularization Techniques

In [ ]:
# L1 and L2 Regularization (Weight Decay)
print("=== Weight Decay (L2 Regularization) ===")

model = nn.Linear(10, 1)

# L2 regularization via weight_decay
optimizer_l2 = optim.SGD(model.parameters(), lr=0.01, weight_decay=0.01)

# Manual L1 regularization
def l1_regularization(model, lambda_l1=0.01):
    l1_loss = 0
    for param in model.parameters():
        l1_loss += torch.sum(torch.abs(param))
    return lambda_l1 * l1_loss

print(f"L2 (weight_decay): Built into optimizer")
print(f"L1: Add to loss manually")

In [ ]:
# Dropout
print("=== Dropout ===")

class NetworkWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(50, 10)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # Only active during training
        x = self.fc2(x)
        return x

model = NetworkWithDropout(0.5)
x = torch.randn(5, 100)

# Training mode (dropout active)
model.train()
out_train = model(x)
print(f"Training - Non-zero ratio: {(out_train != 0).float().mean():.2f}")

# Eval mode (dropout inactive)
model.eval()
out_eval = model(x)
print(f"Eval - Non-zero ratio: {(out_eval != 0).float().mean():.2f}")

In [ ]:
# Batch Normalization
print("=== Batch Normalization ===")

class NetworkWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.bn1 = nn.BatchNorm1d(50)
        self.fc2 = nn.Linear(50, 10)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

model = NetworkWithBatchNorm()
x = torch.randn(32, 100)  # Needs batch size > 1
out = model(x)

print(f"BatchNorm1d tracks:")
print(f"  - Running mean: {model.bn1.running_mean.shape}")
print(f"  - Running var: {model.bn1.running_var.shape}")

## 2. Optimizers Comparison

In [ ]:
# Compare optimizers on simple function
def rosenbrock(x, y):
    """Rosenbrock function - minimum at (1, 1)."""
    return (1 - x)**2 + 100 * (y - x**2)**2

def optimize_trajectory(optimizer_class, **kwargs):
    """Run optimization and return trajectory."""
    x = torch.tensor([-1.0], requires_grad=True)
    y = torch.tensor([1.0], requires_grad=True)
    
    optimizer = optimizer_class([x, y], **kwargs)
    
    trajectory = [(x.item(), y.item())]
    
    for _ in range(500):
        optimizer.zero_grad()
        loss = rosenbrock(x, y)
        loss.backward()
        optimizer.step()
        trajectory.append((x.item(), y.item()))
    
    return trajectory

# Run different optimizers
trajectories = {
    'SGD': optimize_trajectory(optim.SGD, lr=0.001),
    'SGD + Momentum': optimize_trajectory(optim.SGD, lr=0.001, momentum=0.9),
    'Adam': optimize_trajectory(optim.Adam, lr=0.01),
    'RMSprop': optimize_trajectory(optim.RMSprop, lr=0.01)
}

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Contour plot of Rosenbrock
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = rosenbrock(torch.tensor(X), torch.tensor(Y)).numpy()

ax.contour(X, Y, np.log(Z + 1), levels=30, cmap='viridis', alpha=0.5)

colors = ['red', 'blue', 'green', 'orange']
for (name, traj), color in zip(trajectories.items(), colors):
    traj = np.array(traj)
    ax.plot(traj[:, 0], traj[:, 1], '-o', label=name, color=color, 
            markersize=1, alpha=0.7)

ax.scatter([1], [1], color='black', s=100, marker='*', label='Minimum', zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Optimizer Comparison on Rosenbrock Function')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

optimizers_df = pd.DataFrame({
    'Optimizer': ['SGD', 'SGD+Momentum', 'Adam', 'AdamW', 'RMSprop'],
    'Adaptive LR': ['No', 'No', 'Yes', 'Yes', 'Yes'],
    'Momentum': ['No', 'Yes', 'Yes', 'Yes', 'No'],
    'Weight Decay': ['Standard', 'Standard', 'L2', 'Decoupled', 'Standard'],
    'Best For': ['Simple', 'CNNs', 'General', 'Transformers', 'RNNs']
})
print(optimizers_df.to_string(index=False))

## 3. Learning Rate Schedules

In [ ]:
# Learning rate schedulers
model = nn.Linear(10, 1)

schedulers = {}

# Step LR
opt = optim.SGD(model.parameters(), lr=0.1)
schedulers['StepLR'] = optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1)

# Exponential LR
opt = optim.SGD(model.parameters(), lr=0.1)
schedulers['ExponentialLR'] = optim.lr_scheduler.ExponentialLR(opt, gamma=0.95)

# Cosine Annealing
opt = optim.SGD(model.parameters(), lr=0.1)
schedulers['CosineAnnealingLR'] = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

# Reduce on Plateau (needs loss value)
opt = optim.SGD(model.parameters(), lr=0.1)
schedulers['ReduceLROnPlateau'] = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10)

# Plot schedules
fig, ax = plt.subplots(figsize=(12, 6))

epochs = 100
for name, scheduler in schedulers.items():
    if name == 'ReduceLROnPlateau':
        continue  # Needs validation loss
    
    lrs = []
    for _ in range(epochs):
        lrs.append(scheduler.get_last_lr()[0])
        scheduler.step()
    
    ax.plot(lrs, label=name)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# OneCycleLR - popular for fast training
model = nn.Linear(10, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Simulating 100 epochs with 10 batches each
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=0.1, 
    epochs=100, 
    steps_per_epoch=10
)

lrs = []
for epoch in range(100):
    for batch in range(10):
        lrs.append(scheduler.get_last_lr()[0])
        scheduler.step()

plt.figure(figsize=(12, 4))
plt.plot(lrs)
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('OneCycleLR Schedule')
plt.show()

## 4. Early Stopping

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    
    def __init__(self, patience=7, min_delta=0, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.counter = 0
        self.best_loss = None
        self.best_weights = None
        self.early_stop = False
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self._save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.restore_best:
                    model.load_state_dict(self.best_weights)
        else:
            self.best_loss = val_loss
            self._save_checkpoint(model)
            self.counter = 0
    
    def _save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

print("EarlyStopping implementation:")
print("- Tracks validation loss")
print("- Stops when no improvement for 'patience' epochs")
print("- Can restore best weights")

## 5. Weight Initialization

In [ ]:
# Weight initialization methods
def init_weights(model, method='xavier'):
    """Initialize weights using different methods."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if method == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            elif method == 'kaiming':
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            elif method == 'normal':
                nn.init.normal_(m.weight, mean=0, std=0.01)
            
            if m.bias is not None:
                nn.init.zeros_(m.bias)

# Compare initializations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

methods = ['xavier', 'kaiming', 'normal']
for ax, method in zip(axes, methods):
    model = nn.Linear(100, 100)
    init_weights(model, method)
    
    weights = model.weight.data.flatten().numpy()
    ax.hist(weights, bins=50, edgecolor='black')
    ax.set_title(f'{method.capitalize()} Init\nstd={weights.std():.4f}')
    ax.set_xlabel('Weight Value')

plt.tight_layout()
plt.show()

## 6. Gradient Clipping

In [ ]:
# Gradient clipping
model = nn.Sequential(
    nn.Linear(10, 50),
    nn.ReLU(),
    nn.Linear(50, 1)
)

optimizer = optim.Adam(model.parameters())
x = torch.randn(32, 10)
y = torch.randn(32, 1)

# Forward + backward
output = model(x)
loss = F.mse_loss(output, y)
loss.backward()

# Before clipping
total_norm_before = torch.sqrt(sum(p.grad.norm()**2 for p in model.parameters()))
print(f"Gradient norm before clipping: {total_norm_before:.4f}")

# Clip gradients
max_norm = 1.0
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

total_norm_after = torch.sqrt(sum(p.grad.norm()**2 for p in model.parameters()))
print(f"Gradient norm after clipping: {total_norm_after:.4f}")

## 7. Mixed Precision Training

In [ ]:
# Mixed precision training (for GPU)
print("=== Mixed Precision Training ===")
print("Uses float16 for faster computation while maintaining float32 for stability.")
print()

# Example usage (requires GPU)
code_example = '''
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for data, target in dataloader:
    optimizer.zero_grad()
    
    # Forward pass with autocasting
    with autocast():
        output = model(data)
        loss = criterion(output, target)
    
    # Backward pass with scaling
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
'''
print("Example code:")
print(code_example)

## 8. Model Checkpointing

In [ ]:
# Save and load checkpoints
model = nn.Linear(10, 1)
optimizer = optim.Adam(model.parameters())

# Save checkpoint
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.5,
}
# torch.save(checkpoint, 'checkpoint.pth')

# Load checkpoint
# checkpoint = torch.load('checkpoint.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# epoch = checkpoint['epoch']
# loss = checkpoint['loss']

print("Checkpoint includes:")
for key in checkpoint.keys():
    print(f"  - {key}")

## 9. Training Best Practices Summary

In [ ]:
best_practices = pd.DataFrame({
    'Technique': [
        'Batch Normalization',
        'Dropout',
        'Weight Decay',
        'Learning Rate Schedule',
        'Early Stopping',
        'Gradient Clipping',
        'Data Augmentation',
        'Mixed Precision'
    ],
    'When to Use': [
        'Almost always (before activation)',
        'After dense layers (0.2-0.5)',
        'To prevent overfitting (0.0001-0.01)',
        'When loss plateaus',
        'When validation loss increases',
        'With RNNs/Transformers',
        'Limited training data',
        'Large models on GPU'
    ]
})

print(best_practices.to_string(index=False))

## 10. Key Takeaways

1. **Regularization** prevents overfitting (dropout, weight decay, batch norm)
2. **Adam** is a good default optimizer; AdamW for transformers
3. **Learning rate schedules** improve convergence (cosine annealing, one-cycle)
4. **Early stopping** prevents overtraining
5. **Proper initialization** (Xavier/Kaiming) speeds up training
6. **Gradient clipping** stabilizes training for RNNs
7. **Checkpointing** enables training resumption
8. **Mixed precision** speeds up GPU training

### Training Recipe
1. Start with Adam (lr=1e-3)
2. Add batch normalization
3. Use dropout (0.2-0.5)
4. Implement early stopping
5. Add learning rate schedule
6. Tune hyperparameters